In [2]:
import os 
import pandas as pd
import numpy as np
cwd = os.getcwd()
sep = os.path.sep
cwd

'/home/ugo/Scrivania/Transthyretin/experimental_validation'

In [3]:
df_experimental = pd.read_csv(cwd + sep + "experimental_affinity.csv")
df_experimental

,Mutation,Tafamidis,Acoramidis,Diflunisal,Tolcapone
0,WildType,-11.39,-11.34,NaN,-10.5
1,V142I,NaN,NaN,NaN,-10.0
2,V50M,NaN,NaN,NaN,-8.7


In [4]:
dict_experimental = {}
for index, row in df_experimental.iterrows():  
    mutation = row["Mutation"]
    dict_experimental[mutation] = {}
    for ligand in df_experimental.columns[1:]:
        dict_experimental[mutation][ligand] = float(row[ligand])
dict_experimental

{'WildType': {'Tafamidis': -11.39,
  'Acoramidis': -11.34,
  'Diflunisal': nan,
  'Tolcapone': -10.5},
 'V142I': {'Tafamidis': nan,
  'Acoramidis': nan,
  'Diflunisal': nan,
  'Tolcapone': -10.0},
 'V50M': {'Tafamidis': nan,
  'Acoramidis': nan,
  'Diflunisal': nan,
  'Tolcapone': -8.7}}

In [27]:
parent_dir = os.path.dirname(cwd)
docking_folder = parent_dir + sep + "docking" + sep + "AutoDock" + sep + "docked-outputs" + sep + "existing"
filename = "vina_score_docking_{}_TTRtetramer.csv"
df_autodock_acora = pd.read_csv(docking_folder + sep + filename.format("acoramidis"))
df_autodock_tafa = pd.read_csv(docking_folder + sep + filename.format("tafamidis"))
df_autodock_diflu = pd.read_csv(docking_folder + sep + filename.format("diflunisal"))
df_autodock_tolca = pd.read_csv(docking_folder + sep + filename.format("tolcapone"))

df_autodock_acora["Mutation"] = [mutation.split("-")[0].upper() for mutation in df_autodock_acora["Mutation"]]
df_autodock_tafa["Mutation"] = [mutation.split("-")[0].upper() for mutation in df_autodock_tafa["Mutation"]]
df_autodock_diflu["Mutation"] = [mutation.split("-")[0].upper() for mutation in df_autodock_diflu["Mutation"]]
df_autodock_tolca["Mutation"] = [mutation.split("-")[0].upper() for mutation in df_autodock_tolca["Mutation"]]

df_autodock_acora["Mutation"] = [mutation if mutation != "WT" else "WildType" for mutation in df_autodock_acora["Mutation"]]
df_autodock_tafa["Mutation"] = [mutation if mutation != "WT" else "WildType" for mutation in df_autodock_tafa["Mutation"]]
df_autodock_diflu["Mutation"] = [mutation if mutation != "WT" else "WildType" for mutation in df_autodock_diflu["Mutation"]]
df_autodock_tolca["Mutation"] = [mutation if mutation != "WT" else "WildType" for mutation in df_autodock_tolca["Mutation"]]

columns_to_drop = ["Score_2", "Score_3", "Score_4", "Score_5", "Score_6", "Score_7", "Score_8", "Score_9"]
df_autodock_acora.drop(columns=columns_to_drop, inplace=True)
df_autodock_tafa.drop(columns=columns_to_drop, inplace=True)
df_autodock_diflu.drop(columns=columns_to_drop, inplace=True)
df_autodock_tolca.drop(columns=columns_to_drop, inplace=True)   

df_autodock_acora.rename(columns={"Score_1": "Vinardo Score"}, inplace=True)
df_autodock_tafa.rename(columns={"Score_1": "Vinardo Score"}, inplace=True)
df_autodock_diflu.rename(columns={"Score_1": "Vinardo Score"}, inplace=True)
df_autodock_tolca.rename(columns={"Score_1": "Vinardo Score"}, inplace=True)

df_autodock_acora

,Mutation,Vinardo Score
0,A129V,-5.7448
1,V40I,-5.7956
2,D59V,-5.5582
3,E62G,-5.6491
4,A65S,-5.7069
...,...,...
129,A111S,-5.9752
130,G73E,-2.9681
131,R124C,-5.7308
132,A117G,-5.7162


In [29]:
dict_autodock = {}

for mutation, values in dict_experimental.items():
    dict_autodock[mutation] = {}
    for ligand in values.keys():
        if ligand == "Acoramidis":
            df_autodock = df_autodock_acora
        elif ligand == "Tafamidis":
            df_autodock = df_autodock_tafa
        elif ligand == "Diflunisal":
            df_autodock = df_autodock_diflu
        elif ligand == "Tolcapone":
            df_autodock = df_autodock_tolca
        else:
            continue
        
        if mutation in df_autodock["Mutation"].values:
            dict_autodock[mutation][ligand] = float(df_autodock[df_autodock["Mutation"] == mutation]["Vinardo Score"].values[0])
        else:
            dict_autodock[mutation][ligand] = np.nan

dict_autodock

{'WildType': {'Tafamidis': -4.385,
  'Acoramidis': -5.5579,
  'Diflunisal': -6.97,
  'Tolcapone': -5.261},
 'V142I': {'Tafamidis': -3.2176,
  'Acoramidis': -5.2583,
  'Diflunisal': -7.0786,
  'Tolcapone': -4.7336},
 'V50M': {'Tafamidis': -4.2412,
  'Acoramidis': -5.423,
  'Diflunisal': -6.8759,
  'Tolcapone': -4.718}}

In [30]:
from scipy.stats import pearsonr

values_experimental = []
values_autodock = []

for mutation in dict_experimental:
    if mutation in dict_autodock:
        for ligand in dict_experimental[mutation]:
            if dict_experimental[mutation][ligand] is None or np.isnan(dict_experimental[mutation][ligand]):
                #print(f"Skipping mutation {mutation} because it has no experimental value for {ligand}.")
                continue
            if ligand in dict_autodock[mutation]:
                values_experimental.append(dict_experimental[mutation][ligand])
                values_autodock.append(dict_autodock[mutation][ligand])

print("Experimental values:", values_experimental)
print("AutoDock values:", values_autodock)

correlation, p_value = pearsonr(values_experimental, values_autodock)
correlation, p_value

Experimental values: [-11.39, -11.34, -10.5, -10.0, -8.7]
AutoDock values: [-4.385, -5.5579, -5.261, -4.7336, -4.718]


(0.25057077401345423, 0.6843340407912918)